# GestureX model evaluation

This notebook is the **final analysis notebook** in the GestureX pipeline. It loads the dataset collected with `src/collect_data.py`, verifies session separation, checks class balance, trains candidate models in a session-aware manner, and produces:

1. Class balance analysis
2. Session separation verification
3. Validation-only model comparison (logistic regression, random forest, RBF SVM)
4. The **required four-cell generalization table** (raw vs invariant × same-session vs cross-session)
5. Confusion matrices for both representations
6. Interpretation of the generalization gap

> **Honest results policy:** this notebook does not embed pre-computed outputs. All tables, figures, and numbers are derived from your locally collected data after running the pipeline.

## 0. Setup

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, f1_score


def find_repository_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'src' / 'config.py').is_file():
            return candidate
    raise RuntimeError('Could not find the GestureX repository root.')


ROOT = find_repository_root(Path.cwd().resolve())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config import (
    GESTURE_DISPLAY_NAMES,
    GESTURE_LABELS,
    MODELS_DIR,
    RAW_DATA_DIR,
    RESULTS_DIR,
)
from src.features import transform_feature_frame
from src.preprocess import (
    class_distribution,
    load_dataset,
    split_by_session,
    validate_dataset,
)

DISPLAY = {label: GESTURE_DISPLAY_NAMES.get(label, label) for label in GESTURE_LABELS}
print(f'Repository root: {ROOT}')
print(f'Raw data dir:    {RAW_DATA_DIR}')
print(f'Results dir:     {RESULTS_DIR}')

## 1. Load the collected dataset

We merge all session CSVs from `data/raw/` and validate the schema. A missing or malformed column will raise a clear error here rather than silently producing nonsense results.

In [ ]:
csv_paths = sorted(RAW_DATA_DIR.glob('*.csv'))
if not csv_paths:
    raise FileNotFoundError(
        f'No CSV files found in {RAW_DATA_DIR}. '
        'Run python src/collect_data.py first.'
    )

frames = [pd.read_csv(p) for p in csv_paths]
raw_data = pd.concat(frames, ignore_index=True)
dataset = validate_dataset(raw_data, require_unique_sample_ids=True)

print(f'Total samples   : {len(dataset):,}')
print(f'Sessions present: {sorted(dataset["session_id"].unique())}')
print(f'Subjects present: {sorted(dataset["subject_id"].unique())}')
dataset[['sample_id', 'session_id', 'subject_id', 'gesture', 'timestamp']].head()

## 2. Class balance analysis

An imbalanced dataset can inflate macro accuracy while hiding per-class failure. We inspect counts, percentages, and imbalance ratio before fitting any model.

If class weights or resampling are applied later, they will be documented here.

In [ ]:
dist = class_distribution(dataset)
print(f'Total samples     : {dist.attrs["total_samples"]}')
print(f'Imbalance ratio   : {dist.attrs["imbalance_ratio"]:.2f}  (max/min class count)')
missing_classes = dist.attrs['missing_supported_classes']
if missing_classes:
    print(f'MISSING CLASSES   : {missing_classes}  ← collect more data before training')
display(dist)

fig, ax = plt.subplots(figsize=(9, 4))
display_labels = [DISPLAY[g] for g in dist['gesture']]
bars = ax.bar(display_labels, dist['count'], color=sns.color_palette('muted', len(dist)))
ax.bar_label(bars, padding=3, fontsize=9)
ax.set_xlabel('Gesture class', fontsize=11)
ax.set_ylabel('Sample count', fontsize=11)
ax.set_title('Class distribution across all sessions', fontsize=12)
ax.tick_params(axis='x', rotation=15)
plt.tight_layout()
plt.show()

## 3. Session separation verification

**Leakage risk:** if the same recording session appears in both training and test, adjacent near-identical webcam frames inflate all accuracy estimates. We verify this cannot happen.

Set `CROSS_SESSION` to the session ID you held out as the independent test session.

In [ ]:
# ─── Configure the cross-session test session ────────────────────────────
# Change this to match the session ID you intended as the independent test.
CROSS_SESSION = 'test_s01'  # Example: replace with your actual held-out session ID.
# ─────────────────────────────────────────────────────────────────────────

sessions = sorted(dataset['session_id'].unique())
if CROSS_SESSION not in sessions:
    raise ValueError(
        f'Cross-session ID "{CROSS_SESSION}" not found. '
        f'Available sessions: {sessions}'
    )

dev_sessions = [s for s in sessions if s != CROSS_SESSION]
print(f'Development sessions  : {dev_sessions}')
print(f'Cross-session test    : [{CROSS_SESSION}]')

dev_data = dataset.loc[dataset['session_id'].isin(dev_sessions)]
cross_data = dataset.loc[dataset['session_id'] == CROSS_SESSION]

# Belt-and-suspenders overlap check.
overlap = set(dev_data['sample_id']) & set(cross_data['sample_id'])
assert not overlap, f'Leakage: {len(overlap)} sample IDs appear in both development and cross-session data!'

print(f'\nDevelopment samples   : {len(dev_data):,}')
print(f'Cross-session samples : {len(cross_data):,}')
print('✓ No sample ID overlap between development and cross-session data.')

## 4. Session-aware train / same-session-test split

We use `split_by_session` rather than a naive random split. This prevents adjacent webcam frames from leaking across the train/test boundary within development sessions. The cross-session set remains untouched.

In [ ]:
# For a simple notebook demonstration we use one development session for
# training and another for same-session testing when multiple sessions exist.
if len(dev_sessions) >= 2:
    same_session_test_id = dev_sessions[-1]
    train_session_ids = dev_sessions[:-1]
    train_data = dev_data.loc[dev_data['session_id'].isin(train_session_ids)].copy()
    same_test_data = dev_data.loc[dev_data['session_id'] == same_session_test_id].copy()
    print(f'Train sessions        : {train_session_ids}')
    print(f'Same-session test     : [{same_session_test_id}]')
else:
    # Only one development session — use last 20 % as same-session test.
    # Note: in a real experiment use the time-blocked split from train.py.
    session_id = dev_sessions[0]
    n = len(dev_data)
    n_test = max(1, int(n * 0.20))
    train_data = dev_data.iloc[:n - n_test].copy()
    same_test_data = dev_data.iloc[n - n_test:].copy()
    print(
        f'Only one development session ({session_id}); '
        f'using last {n_test}/{n} rows as same-session test.'
    )

print(f'\nTrain samples         : {len(train_data):,}')
print(f'Same-session test     : {len(same_test_data):,}')
print(f'Cross-session test    : {len(cross_data):,}')

## 5. Model training and validation

We train Logistic Regression, Random Forest, and RBF SVM for both feature representations. Scalers are fitted **inside** each pipeline and **only on training rows** — never on the test or cross-session partitions.

In [ ]:
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score, confusion_matrix,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC


def make_pipelines(random_state: int = 42) -> dict[str, Pipeline]:
    return {
        'logistic_regression': Pipeline([
            ('scaler', StandardScaler()),
            ('model', LogisticRegression(max_iter=2000, random_state=random_state)),
        ]),
        'random_forest': Pipeline([
            ('model', RandomForestClassifier(
                n_estimators=300, random_state=random_state, n_jobs=-1, class_weight='balanced'
            )),
        ]),
        'rbf_svm': Pipeline([
            ('scaler', StandardScaler()),
            ('model', SVC(kernel='rbf', C=3.0, gamma='scale', probability=True, random_state=random_state)),
        ]),
    }


def build_X_y(frame: pd.DataFrame, representation: str):
    feat_frame = transform_feature_frame(frame, representation)
    X = feat_frame.to_numpy(dtype=float)
    y = frame['gesture'].to_numpy(dtype=str)
    return X, y


def evaluate_pipeline(pipeline, X_train, y_train, X_test, y_test):
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    return {
        'accuracy': accuracy_score(y_test, y_pred),
        'macro_f1': f1_score(y_test, y_pred, average='macro', zero_division=0),
        'macro_precision': precision_score(y_test, y_pred, average='macro', zero_division=0),
        'macro_recall': recall_score(y_test, y_pred, average='macro', zero_division=0),
        'y_pred': y_pred,
    }


results = {}  # results[(representation, model_name, split)] = metrics dict
for representation in ('raw', 'invariant'):
    print(f'\n── Representation: {representation.upper()} ───────────────────────────────')
    try:
        X_train, y_train = build_X_y(train_data, representation)
        X_same, y_same   = build_X_y(same_test_data, representation)
        X_cross, y_cross = build_X_y(cross_data, representation)
    except Exception as exc:
        print(f'  Feature extraction failed: {exc}')
        print('  (Uniform-landmark datasets fail invariant extraction — this is expected.)')
        continue

    for model_name, pipeline in make_pipelines().items():
        print(f'  Training {model_name} …', end=' ')
        # ── LEAKAGE NOTE ──────────────────────────────────────────────────
        # pipeline.fit() is called ONLY on X_train, y_train.
        # The StandardScaler inside the pipeline is fitted on training data
        # exclusively. We then apply the fitted pipeline to test sets.
        # ─────────────────────────────────────────────────────────────────
        same_metrics  = evaluate_pipeline(pipeline, X_train, y_train, X_same, y_same)
        cross_metrics = evaluate_pipeline(pipeline, X_train, y_train, X_cross, y_cross)
        results[(representation, model_name, 'same_session')]  = {**same_metrics,  'y_pred_cross': None}
        results[(representation, model_name, 'cross_session')] = {**cross_metrics, 'y_pred_cross': None}
        print(
            f'same-acc={same_metrics["accuracy"]:.3f}  '
            f'cross-acc={cross_metrics["accuracy"]:.3f}'
        )

print('\nTraining complete (scaler fitted on training data only).')

## 6. Validation model comparison table

In [ ]:
rows = []
for (representation, model_name, split), metrics in results.items():
    rows.append({
        'representation': representation,
        'model': model_name,
        'split': split,
        'accuracy': metrics['accuracy'],
        'macro_f1': metrics['macro_f1'],
        'macro_precision': metrics['macro_precision'],
        'macro_recall': metrics['macro_recall'],
    })

summary_df = pd.DataFrame(rows)
pivot = summary_df.pivot_table(
    index=['representation', 'model'],
    columns='split',
    values=['accuracy', 'macro_f1'],
    aggfunc='first',
).round(4)
display(pivot)

## 7. Required four-cell generalization table

This is the central deliverable. It compares the best-performing model for each representation across same-session and cross-session test partitions. Values are derived from actual measured results.

In [ ]:
four_cell_rows = []
for representation in ('raw', 'invariant'):
    same_key  = (representation, 'same_session')
    cross_key = (representation, 'cross_session')

    # Select the best model by same-session accuracy (only development data).
    best_model = None
    best_acc = -1.0
    for model_name in ('logistic_regression', 'random_forest', 'rbf_svm'):
        key = (representation, model_name, 'same_session')
        if key in results:
            acc = results[key]['accuracy']
            if acc > best_acc:
                best_acc = acc
                best_model = model_name

    if best_model is None:
        continue

    same_acc  = results[(representation, best_model, 'same_session')]['accuracy']
    cross_acc = results[(representation, best_model, 'cross_session')]['accuracy']
    same_f1   = results[(representation, best_model, 'same_session')]['macro_f1']
    cross_f1  = results[(representation, best_model, 'cross_session')]['macro_f1']

    four_cell_rows.append({
        'Feature Representation': representation.replace('_', ' ').title(),
        'Best Model': best_model.replace('_', ' ').title(),
        'Same-Session Accuracy': same_acc,
        'Cross-Session Accuracy': cross_acc,
        'Generalization Gap (Accuracy)': same_acc - cross_acc,
        'Same-Session Macro F1': same_f1,
        'Cross-Session Macro F1': cross_f1,
    })

four_cell_df = pd.DataFrame(four_cell_rows)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
four_cell_df.to_csv(RESULTS_DIR / 'generalization_results.csv', index=False)
print('Required four-cell generalization table:')
display(four_cell_df.set_index('Feature Representation').T)

## 8. Confusion matrices

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix


def plot_confusion(y_true, y_pred, title, output_path=None):
    present = sorted(set(y_true) | set(y_pred))
    display_labels = [DISPLAY.get(label, label) for label in present]
    cm = confusion_matrix(y_true, y_pred, labels=present)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True).clip(min=1)
    fig, ax = plt.subplots(figsize=(max(6, len(present)), max(5, len(present))))
    sns.heatmap(
        cm_norm, annot=cm, fmt='d',
        xticklabels=display_labels, yticklabels=display_labels,
        cmap='Blues', vmin=0, vmax=1, ax=ax,
    )
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.set_title(title)
    plt.tight_layout()
    if output_path:
        plt.savefig(output_path, dpi=150)
        print(f'Saved: {output_path}')
    plt.show()


for representation in ('raw', 'invariant'):
    for model_name in ('logistic_regression', 'random_forest', 'rbf_svm'):
        same_key = (representation, model_name, 'same_session')
        cross_key = (representation, model_name, 'cross_session')
        if same_key not in results:
            continue
        _, y_same  = build_X_y(same_test_data, representation)
        _, y_cross = build_X_y(cross_data, representation)
        y_same_pred  = results[same_key]['y_pred']
        y_cross_pred = results[cross_key]['y_pred']

        plot_confusion(
            y_same, y_same_pred,
            f'{representation} / {model_name} — same-session test',
            RESULTS_DIR / f'confusion_matrix_{representation}_{model_name}_same.png',
        )
        plot_confusion(
            y_cross, y_cross_pred,
            f'{representation} / {model_name} — cross-session test',
            RESULTS_DIR / f'confusion_matrix_{representation}_{model_name}_cross.png',
        )

## 9. Generalization gap discussion

The generalization gap is defined as:

```
gap = same_session_accuracy − cross_session_accuracy
```

A smaller gap indicates better generalization. Invariant features are designed to reduce sensitivity to framing and scale, which are the primary sources of cross-session variation. The cell below derives this conclusion from actual measured results — not from assumptions.

In [ ]:
if len(four_cell_rows) >= 2:
    raw_row = [r for r in four_cell_rows if 'Raw' in r['Feature Representation']][0]
    inv_row = [r for r in four_cell_rows if 'Invariant' in r['Feature Representation']][0]

    print('Generalization gap (same-session accuracy − cross-session accuracy):')
    print(f"  Raw coordinates : {raw_row['Generalization Gap (Accuracy)']:.4f}")
    print(f"  Invariant feats : {inv_row['Generalization Gap (Accuracy)']:.4f}")

    if inv_row['Generalization Gap (Accuracy)'] < raw_row['Generalization Gap (Accuracy)']:
        conclusion = (
            'Invariant features produce a SMALLER generalization gap, '
            'suggesting better cross-session robustness.'
        )
    else:
        conclusion = (
            'Raw coordinates produce a smaller gap in this experiment. '
            'Consider whether the cross-session variation in your data is '
            'primarily translation/scale (favours invariant) or appearance-based.'
        )
    print(f'\nConclusion: {conclusion}')

    # Bar chart.
    reps = [r['Feature Representation'] for r in four_cell_rows]
    same_vals  = [r['Same-Session Accuracy']  for r in four_cell_rows]
    cross_vals = [r['Cross-Session Accuracy'] for r in four_cell_rows]
    x = range(len(reps))
    width = 0.35

    fig, ax = plt.subplots(figsize=(7, 4))
    bars1 = ax.bar([xi - width/2 for xi in x], same_vals,  width, label='Same-session',  color='#4c72b0')
    bars2 = ax.bar([xi + width/2 for xi in x], cross_vals, width, label='Cross-session', color='#dd8452')
    ax.bar_label(bars1, fmt='%.3f', padding=3, fontsize=9)
    ax.bar_label(bars2, fmt='%.3f', padding=3, fontsize=9)
    ax.set_xticks(list(x))
    ax.set_xticklabels(reps)
    ax.set_ylim(0, 1.15)
    ax.set_xlabel('Feature representation')
    ax.set_ylabel('Accuracy')
    ax.set_title('Same-session vs cross-session generalization')
    ax.legend()
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'generalization_comparison_notebook.png', dpi=150)
    plt.show()
else:
    print('Insufficient results to compute generalization gap. Ensure both representations ran.')